# Tarea 4 – Actividad 3: Cadena de certificación (ECC)

Cadena de tres niveles con **un solo algoritmo, ECC**: curva `prime256v1` (P-256), firma `ecdsa-with-SHA256` y hash SHA-256, que son los parámetros ya justificados en la Actividad 2. Todo se genera con OpenSSL en el directorio `pki/`.

Raíz (autofirmada, ancla) → Intermedia (`CA:TRUE`, `pathlen:0`) → Servidor (`CA:FALSE`).

## 1. Preparación

In [2]:
%%bash
rm -rf pki && mkdir pki && cd pki
openssl version
# Extensiones de cada nivel
cat > root.ext <<'X'
basicConstraints = critical, CA:TRUE, pathlen:1
keyUsage = critical, keyCertSign, cRLSign
subjectKeyIdentifier = hash
X
cat > inter.ext <<'X'
basicConstraints = critical, CA:TRUE, pathlen:0
keyUsage = critical, keyCertSign, cRLSign
subjectKeyIdentifier = hash
authorityKeyIdentifier = keyid
X
cat > server.ext <<'X'
basicConstraints = critical, CA:FALSE
keyUsage = critical, digitalSignature
extendedKeyUsage = serverAuth
subjectAltName = DNS:telematica.usm.cl
subjectKeyIdentifier = hash
authorityKeyIdentifier = keyid
X
ls

OpenSSL 3.0.13 30 Jan 2024 (Library: OpenSSL 3.0.13 30 Jan 2024)
inter.ext
root.ext
server.ext


## 2. CA raíz (ancla de confianza)

Autofirmada, con `pathlen:1`: permite como máximo **una** CA intermedia debajo. Su llave privada esta en `root.key`

In [3]:
%%bash
cd pki
openssl ecparam -name prime256v1 -genkey -noout -out root.key
cat > root.cnf <<'X'
[req]
distinguished_name = dn
prompt = no
x509_extensions = v3_root
[dn]
C = CL
ST = RM
L = Santiago
O = USM
CN = USM Root CA ECC
[v3_root]
basicConstraints = critical, CA:TRUE, pathlen:1
keyUsage = critical, keyCertSign, cRLSign
subjectKeyIdentifier = hash
X
openssl req -x509 -new -key root.key -sha256 -days 3650 -config root.cnf -out root.pem
openssl x509 -in root.pem -noout -subject -issuer -ext basicConstraints,keyUsage

subject=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
issuer=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
X509v3 Basic Constraints: critical
    CA:TRUE, pathlen:1
X509v3 Key Usage: critical
    Certificate Sign, CRL Sign


## 3. CA intermedia


In [4]:
%%bash
cd pki
openssl ecparam -name prime256v1 -genkey -noout -out inter.key
openssl req -new -key inter.key -sha256 \
  -subj "/C=CL/ST=RM/L=Santiago/O=USM/CN=USM Intermediate CA ECC" -out inter.csr
openssl x509 -req -in inter.csr -CA root.pem -CAkey root.key -sha256 -days 1825 \
  -set_serial 2 -extfile inter.ext -out inter.pem
openssl x509 -in inter.pem -noout -subject -issuer -ext basicConstraints,keyUsage

subject=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
issuer=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
X509v3 Basic Constraints: critical
    CA:TRUE, pathlen:0
X509v3 Key Usage: critical
    Certificate Sign, CRL Sign


Certificate request self-signature ok
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC


## 4. Certificado de servidor

Emitido por la intermedia, con `CA:FALSE`, `digitalSignature` y `serverAuth`. Para el caso 1 se emite además un servidor firmado directamente por la raíz.

In [6]:
%%bash
cd pki

openssl ecparam -name prime256v1 -genkey -noout -out server.key
openssl req -new -key server.key -sha256 \
  -subj "/C=CL/ST=RM/L=Santiago/O=USM/CN=telematica.usm.cl" -out server.csr
openssl x509 -req -in server.csr -CA inter.pem -CAkey inter.key -sha256 -days 365 \
  -set_serial 3 -extfile server.ext -out server.pem

# Servidor firmado por root
openssl ecparam -name prime256v1 -genkey -noout -out server_root.key
openssl req -new -key server_root.key -sha256 \
  -subj "/C=CL/ST=RM/L=Santiago/O=USM/CN=telematica.usm.cl" -out server_root.csr
openssl x509 -req -in server_root.csr -CA root.pem -CAkey root.key -sha256 -days 365 \
  -set_serial 4 -extfile server.ext -out server_root.pem
echo; echo "Archivos:"; ls *.pem


Archivos:
inter.pem
root.pem
server.pem
server_root.pem


Certificate request self-signature ok
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
Certificate request self-signature ok
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl


## 5. Los tres certificados (PEM y texto)

In [7]:
%%bash
cd pki && cat root.pem inter.pem server.pem

-----BEGIN CERTIFICATE-----
MIIB8jCCAZegAwIBAgIUSefabUtphoGsMMVErEUq1X0LYa8wCgYIKoZIzj0EAwIw
VTELMAkGA1UEBhMCQ0wxCzAJBgNVBAgMAlJNMREwDwYDVQQHDAhTYW50aWFnbzEM
MAoGA1UECgwDVVNNMRgwFgYDVQQDDA9VU00gUm9vdCBDQSBFQ0MwHhcNMjYxMTE3
MTg0ODMxWhcNMzYxMTE0MTg0ODMxWjBVMQswCQYDVQQGEwJDTDELMAkGA1UECAwC
Uk0xETAPBgNVBAcMCFNhbnRpYWdvMQwwCgYDVQQKDANVU00xGDAWBgNVBAMMD1VT
TSBSb290IENBIEVDQzBZMBMGByqGSM49AgEGCCqGSM49AwEHA0IABEp+yuBnbtFU
+hYE7JZ/rwxn10V2gJZsrGVq+2qK+S9IOZjuOpVQHgQht31jdnp08vYJFPsgtKSk
KSjond8eQE6jRTBDMBIGA1UdEwEB/wQIMAYBAf8CAQEwDgYDVR0PAQH/BAQDAgEG
MB0GA1UdDgQWBBSfHQilGBqi8QwwmbtRqNi4caM7PjAKBggqhkjOPQQDAgNJADBG
AiEAkDQKHKRVykA27XmY7X254U2N7pVNcbRwvwod+vAk5cACIQDddrRk8Vd5aiAq
9+th9Myj7J+e9C9+QDMhD1NudtSdgA==
-----END CERTIFICATE-----
-----BEGIN CERTIFICATE-----
MIICCDCCAa2gAwIBAgIBAjAKBggqhkjOPQQDAjBVMQswCQYDVQQGEwJDTDELMAkG
A1UECAwCUk0xETAPBgNVBAcMCFNhbnRpYWdvMQwwCgYDVQQKDANVU00xGDAWBgNV
BAMMD1VTTSBSb290IENBIEVDQzAeFw0yNjExMTcxODQ4NTRaFw0zMTExMTYxODQ4
NTRaMF0xCzAJBgNVBAYTAkNMMQswCQYDVQQIDAJS

In [8]:
%%bash
cd pki && openssl x509 -in root.pem -noout -text

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number:
            49:e7:da:6d:4b:69:86:81:ac:30:c5:44:ac:45:2a:d5:7d:0b:61:af
        Signature Algorithm: ecdsa-with-SHA256
        Issuer: C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
        Validity
            Not Before: Nov 17 18:48:31 2026 GMT
            Not After : Nov 14 18:48:31 2036 GMT
        Subject: C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
        Subject Public Key Info:
            Public Key Algorithm: id-ecPublicKey
                Public-Key: (256 bit)
                pub:
                    04:4a:7e:ca:e0:67:6e:d1:54:fa:16:04:ec:96:7f:
                    af:0c:67:d7:45:76:80:96:6c:ac:65:6a:fb:6a:8a:
                    f9:2f:48:39:98:ee:3a:95:50:1e:04:21:b7:7d:63:
                    76:7a:74:f2:f6:09:14:fb:20:b4:a4:a4:29:28:e8:
                    9d:df:1e:40:4e
                ASN1 OID: prime256v1
                NIST CURVE: P-256
        X509v3 extensions:
  

In [9]:
%%bash
cd pki && openssl x509 -in inter.pem -noout -text

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number: 2 (0x2)
        Signature Algorithm: ecdsa-with-SHA256
        Issuer: C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
        Validity
            Not Before: Nov 17 18:48:54 2026 GMT
            Not After : Nov 16 18:48:54 2031 GMT
        Subject: C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
        Subject Public Key Info:
            Public Key Algorithm: id-ecPublicKey
                Public-Key: (256 bit)
                pub:
                    04:d3:d5:77:ae:73:a9:d8:c6:65:28:b8:70:cd:9c:
                    ee:e0:87:4b:73:19:cf:60:5a:4a:b8:b0:80:7b:3b:
                    57:d2:80:fc:23:5d:84:37:a2:49:7b:d3:8e:e1:74:
                    70:3a:d0:ed:ff:9d:9f:19:25:c0:5c:f1:b1:ae:2e:
                    78:f0:c3:d2:86
                ASN1 OID: prime256v1
                NIST CURVE: P-256
        X509v3 extensions:
            X509v3 Basic Constraints: critical
           

In [10]:
%%bash
cd pki && openssl x509 -in server.pem -noout -text

Certificate:
    Data:
        Version: 3 (0x2)
        Serial Number: 3 (0x3)
        Signature Algorithm: ecdsa-with-SHA256
        Issuer: C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
        Validity
            Not Before: Nov 17 18:49:25 2026 GMT
            Not After : Nov 17 18:49:25 2027 GMT
        Subject: C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
        Subject Public Key Info:
            Public Key Algorithm: id-ecPublicKey
                Public-Key: (256 bit)
                pub:
                    04:e4:ac:bf:22:f9:c5:80:10:f9:16:a7:23:46:9b:
                    8b:34:41:64:c1:7d:4b:b6:86:09:e7:3d:d7:3f:99:
                    fc:dd:bc:dd:50:b2:ce:5f:dd:6a:9c:8d:6d:f9:4b:
                    8c:bf:19:16:f5:ee:5e:c1:63:d4:ff:f5:81:59:bd:
                    f0:ec:94:c2:b5
                ASN1 OID: prime256v1
                NIST CURVE: P-256
        X509v3 extensions:
            X509v3 Basic Constraints: critical
         

### Resumen de campos pedidos

Emisor, sujeto, `basicConstraints`, `keyUsage` y `pathlen` de cada nivel (se leen directamente de los certificados).

In [11]:
%%bash
cd pki
for c in root inter server; do
  echo "=== $c.pem ==="
  openssl x509 -in $c.pem -noout -subject -issuer -ext basicConstraints,keyUsage
  openssl x509 -in $c.pem -noout -text | grep -E "Signature Algorithm|ASN1 OID|Public Key Algorithm" | sort -u
  echo
done

=== root.pem ===
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
issuer=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
X509v3 Basic Constraints: critical
    CA:TRUE, pathlen:1
X509v3 Key Usage: critical
    Certificate Sign, CRL Sign
                ASN1 OID: prime256v1
            Public Key Algorithm: id-ecPublicKey
        Signature Algorithm: ecdsa-with-SHA256
    Signature Algorithm: ecdsa-with-SHA256

=== inter.pem ===
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
issuer=C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
X509v3 Basic Constraints: critical
    CA:TRUE, pathlen:0
X509v3 Key Usage: critical
    Certificate Sign, CRL Sign
                ASN1 OID: prime256v1
            Public Key Algorithm: id-ecPublicKey
        Signature Algorithm: ecdsa-with-SHA256
    Signature Algorithm: ecdsa-with-SHA256

=== server.pem ===
subject=C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
issuer=

#Verificacion

**Caso 1.**

In [12]:
%%bash
cd pki && openssl verify -CAfile root.pem server_root.pem

server_root.pem: OK


**Caso 2.**

In [13]:
%%bash
cd pki && openssl verify -CAfile root.pem -untrusted inter.pem server.pem

server.pem: OK


**Caso 3.**

In [14]:
%%bash
cd pki && openssl verify -CAfile root.pem server.pem; echo "código de salida: $?"

código de salida: 2


C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
error 20 at 0 depth lookup: unable to get local issuer certificate
error server.pem: verification failed


**Caso 4.**Cadena completa, pero rompe las restricciones de ruta. Se utilizamos las variantes:

* **4a.** Intermedia sin `keyCertSign` (solo `digitalSignature`).
* **4b.** Un certificado de entidad final (`CA:FALSE`) que se usa como si fuera CA para firmar otro certificado.
* **4c.** Violación de `pathlen`: una CA extra bajo la intermedia, que tiene `pathlen:0`.

In [15]:
%%bash
cd pki
# --- 4a: intermedia mala, sin keyCertSign ---
cat > inter_bad.ext <<'X'
basicConstraints = critical, CA:TRUE, pathlen:0
keyUsage = critical, digitalSignature
subjectKeyIdentifier = hash
authorityKeyIdentifier = keyid
X
openssl ecparam -name prime256v1 -genkey -noout -out inter_bad.key
openssl req -new -key inter_bad.key -sha256 -subj "/C=CL/O=USM/CN=USM Intermediate SIN keyCertSign" -out inter_bad.csr
openssl x509 -req -in inter_bad.csr -CA root.pem -CAkey root.key -sha256 -days 365 -set_serial 5 -extfile inter_bad.ext -out inter_bad.pem 2>/dev/null
openssl req -new -key server.key -sha256 -subj "/C=CL/O=USM/CN=telematica.usm.cl" -out s4a.csr
openssl x509 -req -in s4a.csr -CA inter_bad.pem -CAkey inter_bad.key -sha256 -days 365 -set_serial 6 -extfile server.ext -out s4a.pem 2>/dev/null
echo "--- 4a: intermedia sin keyCertSign ---"
openssl x509 -in inter_bad.pem -noout -ext keyUsage
openssl verify -CAfile root.pem -untrusted inter_bad.pem s4a.pem; echo "código de salida: $?"

--- 4a: intermedia sin keyCertSign ---
X509v3 Key Usage: critical
    Digital Signature
código de salida: 2


C = CL, O = USM, CN = USM Intermediate SIN keyCertSign
error 79 at 1 depth lookup: invalid CA certificate
C = CL, O = USM, CN = USM Intermediate SIN keyCertSign
error 32 at 1 depth lookup: key usage does not include certificate signing
error s4a.pem: verification failed


In [16]:
%%bash
cd pki
# --- 4b: el servidor (CA:FALSE) firma a otro certificado ---
openssl ecparam -name prime256v1 -genkey -noout -out victima.key
openssl req -new -key victima.key -sha256 -subj "/C=CL/O=USM/CN=victima.usm.cl" -out victima.csr
openssl x509 -req -in victima.csr -CA server.pem -CAkey server.key -sha256 -days 30 -set_serial 7 -extfile server.ext -out victima.pem 2>/dev/null
echo "--- 4b: entidad final actuando como CA ---"
openssl x509 -in server.pem -noout -ext basicConstraints
openssl verify -CAfile root.pem -untrusted inter.pem -untrusted server.pem victima.pem; echo "código de salida: $?"

--- 4b: entidad final actuando como CA ---
X509v3 Basic Constraints: critical
    CA:FALSE
código de salida: 2


C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
error 79 at 1 depth lookup: invalid CA certificate
C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
error 25 at 2 depth lookup: path length constraint exceeded
C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
error 25 at 3 depth lookup: path length constraint exceeded
C = CL, ST = RM, L = Santiago, O = USM, CN = telematica.usm.cl
error 32 at 1 depth lookup: key usage does not include certificate signing
error victima.pem: verification failed


In [17]:
%%bash
cd pki
# --- 4c: pathlen excedido (root pathlen:1 -> inter pathlen:0 -> subCA -> servidor) ---
cat > sub.ext <<'X'
basicConstraints = critical, CA:TRUE
keyUsage = critical, keyCertSign, cRLSign
subjectKeyIdentifier = hash
authorityKeyIdentifier = keyid
X
openssl ecparam -name prime256v1 -genkey -noout -out sub.key
openssl req -new -key sub.key -sha256 -subj "/C=CL/O=USM/CN=USM Sub CA (no permitida)" -out sub.csr
openssl x509 -req -in sub.csr -CA inter.pem -CAkey inter.key -sha256 -days 365 -set_serial 8 -extfile sub.ext -out sub.pem 2>/dev/null
openssl req -new -key server.key -sha256 -subj "/C=CL/O=USM/CN=telematica.usm.cl" -out s4c.csr
openssl x509 -req -in s4c.csr -CA sub.pem -CAkey sub.key -sha256 -days 365 -set_serial 9 -extfile server.ext -out s4c.pem 2>/dev/null
echo "--- 4c: pathlen excedido ---"
openssl x509 -in inter.pem -noout -ext basicConstraints
openssl verify -CAfile root.pem -untrusted inter.pem -untrusted sub.pem s4c.pem; echo "código de salida: $?"

--- 4c: pathlen excedido ---
X509v3 Basic Constraints: critical
    CA:TRUE, pathlen:0
código de salida: 2


C = CL, ST = RM, L = Santiago, O = USM, CN = USM Intermediate CA ECC
error 25 at 2 depth lookup: path length constraint exceeded
C = CL, ST = RM, L = Santiago, O = USM, CN = USM Root CA ECC
error 25 at 3 depth lookup: path length constraint exceeded
error s4c.pem: verification failed


## 7. Resumen de resultados

In [18]:
%%bash
cd pki
r() { out=$("$@" 2>&1); if echo "$out" | grep -q ": OK$"; then echo "VÁLIDO"; else echo "INVÁLIDO  ->  $(echo "$out" | grep -m1 error)"; fi; }
printf "%-4s %s\n" "1"  "$(r openssl verify -CAfile root.pem server_root.pem)"
printf "%-4s %s\n" "2"  "$(r openssl verify -CAfile root.pem -untrusted inter.pem server.pem)"
printf "%-4s %s\n" "3"  "$(r openssl verify -CAfile root.pem server.pem)"
printf "%-4s %s\n" "4a" "$(r openssl verify -CAfile root.pem -untrusted inter_bad.pem s4a.pem)"
printf "%-4s %s\n" "4b" "$(r openssl verify -CAfile root.pem -untrusted inter.pem -untrusted server.pem victima.pem)"
printf "%-4s %s\n" "4c" "$(r openssl verify -CAfile root.pem -untrusted inter.pem -untrusted sub.pem s4c.pem)"

1    VÁLIDO
2    VÁLIDO
3    INVÁLIDO  ->  error 20 at 0 depth lookup: unable to get local issuer certificate
4a   INVÁLIDO  ->  error 79 at 1 depth lookup: invalid CA certificate
4b   INVÁLIDO  ->  error 79 at 1 depth lookup: invalid CA certificate
4c   INVÁLIDO  ->  error 25 at 2 depth lookup: path length constraint exceeded


## 8. Explicación(es)
La raiz es el ancla al ser el unico certificado que el verificador posee de antemano, y es autofirmada al no haber nadie por encima que la avale por lo que su confianza es configurada. El intermedio es el certificado que viaja en la cadena junto al servidor, el servidor muestra su certificado y la intermedia y con ello el cliente reconstruye la ruta hasta el ancla. El certificado del servidor es la hoja CA:False y no emite nada.

 Si el cliente aceptara la raíz que viene en la propia conexión, cualquier atacante podría presentar su propia raíz autofirmada y toda la verificación sería circular. Por lo tanto el certificado nunca deberia pasar a ser ancla solo por ser enviado por el servidor.

 Si la intermedia se compromete, se revoca o se rota sin tocar las raices instaladas en los clientes el verificador solo vera una firma de un desconocido y la ruta nose podra construir, pese a que el certificado sea legitimo, como ocurre en el caso 3.

La cadena criptográficamente puede estar bien (las firmas son válidas) y aun así fallar por restricciones de ruta de RFC 5280: una CA debe tener `keyCertSign` para firmar (4a), un certificado `CA:FALSE` no puede actuar como emisor (4b) y `pathlen` limita cuántas CA pueden colgar bajo una CA dada (4c). Verificar la firma es necesario pero no suficiente.

La cadena de certificados establece confianza en la identidad y la llave pública del servidor; no cifra la sesión, el cifrado lo hace TLS con las llaves de sesión que se negocian después.